Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [4]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(43800, 6)

In [10]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 24
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 6)
Dimensiones de Y: (43765, 1)


In [15]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [16]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (43765, 72)


Se dividen nuevamente los conjuntos de datos

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 72)
Las dimensiones de testX son:  (8797, 72)
Las dimensiones de valX son:  (4333, 72)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

240/240 - 7s - 30ms/step - ia: 0.2652 - loss: 1.1480 - mae: 0.7886 - rmse: 1.0648 - smape: 1.4519 - val_ia: 0.2527 - val_loss: 0.9630 - val_mae: 0.7286 - val_rmse: 0.8720 - val_smape: 1.8310

Epoch 2/128                                           

240/240 - 2s - 7ms/step - ia: 0.2256 - loss: 1.0978 - mae: 0.7840 - rmse: 1.0416 - smape: 1.5193 - val_ia: 0.2533 - val_loss: 0.9613 - val_mae: 0.7323 - val_rmse: 0.8743 - val_smape: 1.9187

Epoch 3/128                                           

240/240 - 1s - 4ms/step - ia: 0.2040 - loss: 1.0675 - mae: 0.7753 - rmse: 1.0279 - smape: 1.5541 - val_ia: 0.2537 - val_loss: 0.9592 - val_mae: 0.7321 - val_rmse: 0.8738 - val_smape: 1.9151

Epoch 4/128                                           

240/240 - 1s - 3ms/step - ia: 0.1898 - loss: 1.0444 - mae: 0.7681 - rmse: 1.0153 - smape: 1.5832 - val_ia: 0.2541 - val_loss: 0.9563 - val_mae: 0.7300 - val_rmse: 0.8718 - val_smape: 1.8867

Epoch 5/128

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                      

1915/1915 - 14s - 7ms/step - ia: 0.3788 - loss: 0.8591 - mae: 0.6829 - rmse: 0.8937 - smape: 1.3029 - val_ia: 0.2594 - val_loss: 0.7753 - val_mae: 0.6466 - val_rmse: 0.7226 - val_smape: 1.2219

Epoch 2/128                                                                      

1915/1915 - 6s - 3ms/step - ia: 0.3793 - loss: 0.8326 - mae: 0.6714 - rmse: 0.8804 - smape: 1.2979 - val_ia: 0.2475 - val_loss: 0.7687 - val_mae: 0.6501 - val_rmse: 0.7160 - val_smape: 1.3289

Epoch 3/128                                                                      

1915/1915 - 6s - 3ms/step - ia: 0.3825 - loss: 0.8284 - mae: 0.6690 - rmse: 0.8765 - smape: 1.2946 - val_ia: 0.2580 - val_loss: 0.7553 - val_mae: 0.6358 - val_rmse: 0.7102 - val_smape: 1.2111

Epoch 4/128                                                                      

1915/1915 - 7s - 4ms/step - ia: 0.3799 - loss: 0.8248 - mae: 0.6687 - rmse: 0.8762 - sma

...

Epoch 1/128                                                                      

1915/1915 - 13s - 7ms/step - ia: 0.3338 - loss: 1.1604 - mae: 0.8170 - rmse: 1.0440 - smape: 1.3763 - val_ia: 0.2378 - val_loss: 0.7976 - val_mae: 0.6624 - val_rmse: 0.7257 - val_smape: 1.3492

Epoch 2/128                                                                      

1915/1915 - 9s - 5ms/step - ia: 0.3546 - loss: 0.9526 - mae: 0.7302 - rmse: 0.9449 - smape: 1.3427 - val_ia: 0.2408 - val_loss: 0.7833 - val_mae: 0.6585 - val_rmse: 0.7234 - val_smape: 1.3211

Epoch 3/128                                                                      

1915/1915 - 10s - 5ms/step - ia: 0.3620 - loss: 0.8900 - mae: 0.7019 - rmse: 0.9123 - smape: 1.3284 - val_ia: 0.2441 - val_loss: 0.7703 - val_mae: 0.6490 - val_rmse: 0.7138 - val_smape: 1.3015

Epoch 4/128                                                                      

1915/1915 - 11s - 6ms/step - ia: 0.3676 - loss: 0.8619 - mae: 0.6878 - rmse: 0.8958 - s

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                         

3830/3830 - 16s - 4ms/step - ia: 0.2458 - loss: 1.9123 - mae: 1.0295 - rmse: 1.2964 - smape: 1.6086 - val_ia: 0.1666 - val_loss: 1.2150 - val_mae: 0.8835 - val_rmse: 0.9259 - val_smape: 1.7227

Epoch 2/128                                                                         

3830/3830 - 8s - 2ms/step - ia: 0.2500 - loss: 1.8187 - mae: 1.0073 - rmse: 1.2663 - smape: 1.6074 - val_ia: 0.1680 - val_loss: 1.1826 - val_mae: 0.8670 - val_rmse: 0.9091 - val_smape: 1.7288

Epoch 3/128                                                                         

3830/3830 - 10s - 3ms/step - ia: 0.2487 - loss: 1.6930 - mae: 0.9819 - rmse: 1.2293 - smape: 1.6083 - val_ia: 0.1693 - val_loss: 1.1546 - val_mae: 0.8526 - val_rmse: 0.8944 - val_smape: 1.7350

Epoch 4/128                                                                         

3830/3830 - 11s - 3ms/step - ia: 0.2478 - loss: 1.6485 - mae: 0.9668 - rmse

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                         

240/240 - 3s - 14ms/step - ia: 0.3726 - loss: 0.9955 - mae: 0.7475 - rmse: 0.9923 - smape: 1.3176 - val_ia: 0.3279 - val_loss: 0.7737 - val_mae: 0.6497 - val_rmse: 0.7975 - val_smape: 1.2757

Epoch 2/128                                                                         

240/240 - 1s - 3ms/step - ia: 0.3878 - loss: 0.9439 - mae: 0.7263 - rmse: 0.9670 - smape: 1.2997 - val_ia: 0.3292 - val_loss: 0.7576 - val_mae: 0.6409 - val_rmse: 0.7871 - val_smape: 1.2604

Epoch 3/128                                                                         

240/240 - 1s - 2ms/step - ia: 0.3941 - loss: 0.9062 - mae: 0.7101 - rmse: 0.9472 - smape: 1.2882 - val_ia: 0.3295 - val_loss: 0.7593 - val_mae: 0.6441 - val_rmse: 0.7888 - val_smape: 1.2845

Epoch 4/128                                                                         

240/240 - 1s - 2ms/step - ia: 0.3916 - loss: 0.8974 - mae: 0.7064 - rmse: 0.9420 -

In [23]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}
